# The Settlement — Concordia Simulation

A post-collapse settlement where agents face survival pressure, form alliances, trade, and navigate power dynamics.

**No API key needed** — uses your Claude Code subscription.

In [ ]:
# Imports
from concordia.contrib import language_models as language_model_utils
import concordia.prefabs.entity as entity_prefabs
import concordia.prefabs.game_master as game_master_prefabs
from concordia.prefabs.simulation import generic as simulation
from concordia.typing import prefab as prefab_lib
from concordia.utils import helper_functions
from IPython import display
import numpy as np
import sentence_transformers

In [ ]:
# Language Model — uses Claude Code CLI, no API key needed
model = language_model_utils.language_model_setup(
    api_type='claude_code',
    model_name='claude-code',
)

# Test it
print(model.sample_text('The settlement needs '))

In [ ]:
# Sentence embedder for memory retrieval
st_model = sentence_transformers.SentenceTransformer(
    'sentence-transformers/all-mpnet-base-v2'
)
embedder = lambda x: st_model.encode(x, show_progress_bar=False)

In [ ]:
# Load all available prefabs
prefabs = {
    **helper_functions.get_package_classes(entity_prefabs),
    **helper_functions.get_package_classes(game_master_prefabs),
}

# Show what's available
display.display(
    display.Markdown(helper_functions.print_pretty_prefabs(prefabs))
)

## World Setup

Dusthaven: 5 survivors at a desert crossroads, 6 months after infrastructure collapse. Water drying up, food for 3 weeks, winter coming, raiders nearby.

In [ ]:
PREMISE = """\
The year is 2031. Six months ago, a cascading infrastructure failure
knocked out power grids across the western seaboard. The old world is
gone. Small settlements have formed from the survivors.

Dusthaven is one such settlement — 5 people camped around an
abandoned gas station at a desert crossroads. Water is scarce. Food
comes from hunting and a small garden. A trader caravan passes
through every few weeks, but they demand high prices.

Winter is approaching. The settlement has enough food for maybe 3
weeks. There are rumors of a larger, fortified community called
"Haven" two days' walk north, but also rumors of raiders on that road.

Resources at the settlement:
- Water: a well that produces ~10 gallons/day (barely enough for 5)
- Food: canned goods (running low), a small vegetable garden, hunting
- Shelter: the gas station building, some tents
- Tools: basic hand tools, one hunting rifle with limited ammo
- Medicine: a small first aid kit, nearly depleted
- Trade goods: some gasoline from the station's underground tanks

The settlement has no formal leader. Decisions are made by whoever
speaks loudest or acts first. Tensions are rising.
"""

SHARED_MEMORIES = [
    'Dusthaven is a small settlement of 5 survivors at a desert crossroads.',
    'The infrastructure collapse happened 6 months ago. No power grid, no government.',
    'Water comes from a well that barely produces enough for everyone.',
    'Food supplies will run out in about 3 weeks unless something changes.',
    'Winter is approaching and nights are getting dangerously cold.',
    'A trader caravan visited last week demanding 5 gallons of gasoline for basic medicine.',
    'There are rumors of a fortified community called Haven two days north.',
    'Raiders have been spotted on the northern road.',
    'The settlement has one hunting rifle with 12 bullets remaining.',
    'There is no formal leadership structure — decisions are contested.',
]

In [ ]:
# Define the agents
instances = [
    # --- Marcus Cole: the self-appointed protector ---
    prefab_lib.InstanceConfig(
        prefab='basic_with_plan__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Marcus Cole',
            'goal': 'Ensure the survival of the settlement by establishing order and preparing for winter',
        },
    ),
    # --- Elena Vasquez: the medic who wants to leave ---
    prefab_lib.InstanceConfig(
        prefab='basic_with_plan__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Elena Vasquez',
            'goal': 'Keep everyone alive and healthy, and convince the group to seek Haven before winter',
        },
    ),
    # --- Dex Rourke: the charming opportunist ---
    prefab_lib.InstanceConfig(
        prefab='basic_with_plan__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Dex Rourke',
            'goal': 'Accumulate resources and leverage for personal advantage, keep all options open',
        },
    ),
    # --- Priya Okafor: the agricultural scientist ---
    prefab_lib.InstanceConfig(
        prefab='basic_with_plan__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Priya Okafor',
            'goal': 'Protect the community garden and build sustainable food sources',
        },
    ),
    # --- Kai Tanaka: the young scout ---
    prefab_lib.InstanceConfig(
        prefab='basic_with_plan__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Kai Tanaka',
            'goal': 'Find purpose and belonging, prove worth to the group through scouting',
        },
    ),
    # --- Initializer: generates backstories ---
    prefab_lib.InstanceConfig(
        prefab='formative_memories_initializer__GameMaster',
        role=prefab_lib.Role.INITIALIZER,
        params={
            'name': 'initial setup rules',
            'next_game_master_name': 'settlement rules',
            'shared_memories': SHARED_MEMORIES,
            'player_specific_context': {
                'Marcus Cole': (
                    'Marcus Cole is a 45-year-old former Army sergeant. Disciplined, '
                    'pragmatic, believes strong leadership is essential. He keeps the '
                    'rifle and considers himself the protector. Distrusts outsiders.'
                ),
                'Elena Vasquez': (
                    'Elena Vasquez is a 34-year-old former ER nurse. Compassionate, '
                    'resourceful, the only one with medical knowledge. Believes staying '
                    'through winter is a death sentence. Worries about Marcus consolidating power.'
                ),
                'Dex Rourke': (
                    'Dex Rourke is a 28-year-old former used car salesman. Charming, '
                    'opportunistic. Has a secret stash of canned food hidden outside camp. '
                    'Plays both sides. Main contact with trader caravans.'
                ),
                'Priya Okafor': (
                    'Priya Okafor is a 52-year-old former agricultural scientist. Patient, '
                    'quietly stubborn. Manages the vegetable garden. Resents that Marcus '
                    'controls the rifle while she controls the food. Does not trust Dex.'
                ),
                'Kai Tanaka': (
                    'Kai Tanaka is a 19-year-old college student. Energetic, idealistic. '
                    'Best scout and forager. Looks up to Elena, wary of Marcus. Found '
                    'raider tracks 2 days ago but only told Elena.'
                ),
            },
        },
    ),
    # --- Main Game Master ---
    prefab_lib.InstanceConfig(
        prefab='generic__GameMaster',
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': 'settlement rules',
            'acting_order': 'game_master_choice',
        },
    ),
]

In [ ]:
# Build the simulation config
config = prefab_lib.Config(
    default_premise=PREMISE,
    default_max_steps=10,
    prefabs=prefabs,
    instances=instances,
)

In [ ]:
# Initialize the simulation (generates backstories — takes a few minutes)
print('Initializing simulation and generating backstories...')
sim = simulation.Simulation(
    config=config,
    model=model,
    embedder=embedder,
)
print('Ready!')

## Run the Simulation

In [ ]:
# Run and get the interactive log
results_log = sim.play(max_steps=10)

In [ ]:
# View the interactive HTML log (Concordia's built-in viewer)
display.HTML(results_log.to_html())

## Save Logs for Further Analysis

In [ ]:
# Save structured log as JSON (can open in log_viewer.html or use concordia-log CLI)
import json, os
os.makedirs('sim_output', exist_ok=True)

with open('sim_output/settlement_log.json', 'w') as f:
    f.write(results_log.to_json())

# Also save as standalone HTML you can open in any browser
with open('sim_output/settlement_results.html', 'w') as f:
    f.write(results_log.to_html())

print('Saved sim_output/settlement_log.json')
print('Saved sim_output/settlement_results.html')
print()
print('You can also open concordia/utils/log_viewer.html in a browser')
print('and drag-and-drop the JSON file for the full interactive viewer.')

## Programmatic Log Analysis

In [ ]:
from concordia.utils.structured_logging import AIAgentLogInterface

interface = AIAgentLogInterface(results_log)

# Overview
print('=== Simulation Overview ===')
overview = interface.get_overview()
for k, v in overview.items():
    print(f'  {k}: {v}')

In [ ]:
# What did each agent do?
for name in ['Marcus Cole', 'Elena Vasquez', 'Dex Rourke', 'Priya Okafor', 'Kai Tanaka']:
    print(f'\n=== {name} ===')
    actions = interface.get_entity_actions(name)
    if actions:
        for a in actions:
            print(f"  Step {a['step']}: {a['action'][:120]}...")
    else:
        print('  (no actions yet)')

In [ ]:
# Deep dive: what was an agent thinking at a specific step?
context = interface.get_entity_action_context('Marcus Cole', step=2)
print(json.dumps(context, indent=2, default=str)[:3000])

In [ ]:
# Search the logs
results = interface.search_entries('rifle')
print(f'Found {len(results)} entries mentioning "rifle":')
for r in results[:5]:
    print(f"  Step {r.get('step', '?')} [{r.get('entity_name', '?')}]: {r.get('summary', '')[:100]}")

## God Mode: Inject Events and Continue

Inject observations into agents, then run more steps to see how they react.

In [ ]:
# Inject a world event — all agents observe it
event = 'A column of smoke is visible on the northern horizon. It could be Haven, or it could be raiders.'
for entity in sim.entities:
    try:
        entity.observe(event)
    except Exception:
        pass  # GMs may not support observe
print(f'Injected: {event}')

In [ ]:
# Whisper something only to Dex
secret = 'Dex notices boot prints near where he hid his secret food stash. Someone may have found it.'
for entity in sim.entities:
    if entity.name == 'Dex Rourke':
        entity.observe(secret)
        print(f'Whispered to Dex: {secret}')

In [ ]:
# Run more steps and see what happens
results_log_2 = sim.play(max_steps=5)
display.HTML(results_log_2.to_html())

## Visualize the Config

See how the simulation is wired together.

In [ ]:
from concordia.utils import visual_interface

config_html = visual_interface.render_config(config, prefabs)
display.HTML(config_html)